# Sales Prediction Using Python
Objective: predict product sales from TV, Radio, and Newspaper advertising spend.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
sns.set_theme(style="whitegrid")
df=pd.read_csv("https://raw.githubusercontent.com/selva86/datasets/master/Advertising.csv"); df=df.loc[:,~df.columns.str.startswith("Unnamed")]
print("shape:",df.shape); print("nulls:",df.isnull().sum()); display(df.describe())
sns.pairplot(df); plt.show()
for col in ["TV","radio","newspaper"]: sns.scatterplot(data=df,x=col,y="sales"); plt.title(f"Sales vs {col}"); plt.show()
sns.heatmap(df.corr(numeric_only=True),annot=True,cmap="vlag"); plt.title("Correlation matrix"); plt.show()

In [ ]:
X=df[["TV","radio","newspaper"]]; y=df.sales
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42)
models={"Linear Regression":LinearRegression(),"Random Forest":RandomForestRegressor(n_estimators=200,random_state=42)}; scores={}
for name,model in models.items():
 model.fit(Xtr,ytr); pred=model.predict(Xte); scores[name]={'MAE':mean_absolute_error(yte,pred),'RMSE':mean_squared_error(yte,pred)**.5,'R2':r2_score(yte,pred)}; print(name,scores[name])
model=models[max(scores,key=lambda k:scores[k]['R2'])]; pred=model.predict(Xte)
plt.scatter(yte-yte.mean(),pred-yte.mean()); plt.axline((0,0),slope=1,color="red"); plt.title("Residual diagnostic: predicted vs centered actual"); plt.xlabel("Centered actual"); plt.ylabel("Centered predicted"); plt.show()
if hasattr(model,"coef_"): importance=pd.Series(model.coef_,index=X.columns).abs().sort_values(ascending=False)
else: importance=pd.Series(model.feature_importances_,index=X.columns).sort_values(ascending=False)
print("Channel impact ranking:",importance); print("Highest impact channel:",importance.index[0])